In [1]:
%%writefile parallel_bfs_dfs.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <stack>
#include <omp.h>
using namespace std;

class Graph {
    int V;
    vector<vector<int>> adj;

public:
    Graph(int v) {
        V = v;
        adj.resize(V);
    }

    void addEdge(int u, int v) {
        adj[u].push_back(v);
        adj[v].push_back(u);   // undirected graph
    }

    void printGraph() {
        cout << "\nAdjacency List:\n";
        for (int i = 0; i < V; i++) {
            cout << i << " -> ";
            for (int j : adj[i]) cout << j << " ";
            cout << endl;
        }
    }

    // -------------------------------
    // Sequential BFS
    // -------------------------------
    void sequentialBFS(int start) {
        vector<bool> visited(V, false);
        queue<int> q;

        visited[start] = true;
        q.push(start);

        cout << "\nSequential BFS: ";

        while (!q.empty()) {
            int node = q.front();
            q.pop();

            cout << node << " ";

            for (int nbr : adj[node]) {
                if (!visited[nbr]) {
                    visited[nbr] = true;
                    q.push(nbr);
                }
            }
        }
        cout << endl;
    }

    // -------------------------------
    // Parallel BFS (level-wise)
    // -------------------------------
    void parallelBFS(int start) {
        vector<bool> visited(V, false);
        vector<int> frontier, next_frontier;

        frontier.push_back(start);
        visited[start] = true;

        cout << "\nParallel BFS: ";

        while (!frontier.empty()) {
            next_frontier.clear();

            // Print current frontier
            for (int node : frontier) {
                cout << node << " ";
            }

            #pragma omp parallel
            {
                vector<int> local_next;

                #pragma omp for nowait
                for (int i = 0; i < frontier.size(); i++) {
                    int node = frontier[i];

                    for (int nbr : adj[node]) {
                        bool added = false;

                        #pragma omp critical
                        {
                            if (!visited[nbr]) {
                                visited[nbr] = true;
                                added = true;
                            }
                        }

                        if (added) {
                            local_next.push_back(nbr);
                        }
                    }
                }

                #pragma omp critical
                next_frontier.insert(next_frontier.end(), local_next.begin(), local_next.end());
            }

            frontier = next_frontier;
        }
        cout << endl;
    }

    // -------------------------------
    // Sequential DFS
    // -------------------------------
    void sequentialDFS(int start) {
        vector<bool> visited(V, false);
        stack<int> st;
        st.push(start);

        cout << "\nSequential DFS: ";

        while (!st.empty()) {
            int node = st.top();
            st.pop();

            if (!visited[node]) {
                visited[node] = true;
                cout << node << " ";

                // push neighbors in reverse order for stable output
                for (int i = adj[node].size() - 1; i >= 0; i--) {
                    int nbr = adj[node][i];
                    if (!visited[nbr]) {
                        st.push(nbr);
                    }
                }
            }
        }
        cout << endl;
    }

    // -------------------------------
    // Parallel DFS
    // -------------------------------
    void parallelDFSUtil(int node, vector<bool> &visited) {
        bool doVisit = false;

        #pragma omp critical
        {
            if (!visited[node]) {
                visited[node] = true;
                doVisit = true;
                cout << node << " ";
            }
        }

        if (!doVisit) return;

        #pragma omp parallel for
        for (int i = 0; i < adj[node].size(); i++) {
            int nbr = adj[node][i];

            bool shouldExplore = false;
            #pragma omp critical
            {
                if (!visited[nbr]) {
                    shouldExplore = true;
                }
            }

            if (shouldExplore) {
                parallelDFSUtil(nbr, visited);
            }
        }
    }

    void parallelDFS(int start) {
        vector<bool> visited(V, false);

        cout << "\nParallel DFS: ";

        #pragma omp parallel
        {
            #pragma omp single
            {
                parallelDFSUtil(start, visited);
            }
        }

        cout << endl;
    }
};

int main() {
    int V = 7;
    Graph g(V);

    // Example undirected graph
    g.addEdge(0, 1);
    g.addEdge(0, 2);
    g.addEdge(1, 3);
    g.addEdge(1, 4);
    g.addEdge(2, 5);
    g.addEdge(2, 6);

    g.printGraph();

    int start = 0;

    double start_time, end_time;

    // Sequential BFS
    start_time = omp_get_wtime();
    g.sequentialBFS(start);
    end_time = omp_get_wtime();
    cout << "Sequential BFS Time: " << (end_time - start_time) << " seconds\n";

    // Parallel BFS
    start_time = omp_get_wtime();
    g.parallelBFS(start);
    end_time = omp_get_wtime();
    cout << "Parallel BFS Time: " << (end_time - start_time) << " seconds\n";

    // Sequential DFS
    start_time = omp_get_wtime();
    g.sequentialDFS(start);
    end_time = omp_get_wtime();
    cout << "Sequential DFS Time: " << (end_time - start_time) << " seconds\n";

    // Parallel DFS
    start_time = omp_get_wtime();
    g.parallelDFS(start);
    end_time = omp_get_wtime();
    cout << "Parallel DFS Time: " << (end_time - start_time) << " seconds\n";

    return 0;
}

Writing parallel_bfs_dfs.cpp


In [2]:
!g++ -fopenmp parallel_bfs_dfs.cpp -o parallel_bfs_dfs

In [3]:
!./parallel_bfs_dfs


Adjacency List:
0 -> 1 2 
1 -> 0 3 4 
2 -> 0 5 6 
3 -> 1 
4 -> 1 
5 -> 2 
6 -> 2 

Sequential BFS: 0 1 2 3 4 5 6 
Sequential BFS Time: 1.7781e-05 seconds

Parallel BFS: 0 1 2 3 4 5 6 
Parallel BFS Time: 0.000143698 seconds

Sequential DFS: 0 1 3 4 2 5 6 
Sequential DFS Time: 7.698e-06 seconds

Parallel DFS: 0 1 3 4 2 5 6 
Parallel DFS Time: 2.2046e-05 seconds
